# Implementing backpropagation with NumPy (2/2)

We start by importing the required packages.

In [1]:
import numpy as np

from torchvision import datasets

Now we load the training- and test-datasets from the  torchvision datasets.

In [2]:
X_train = datasets.MNIST("../data", train=True, download=True).data.numpy()
y_train = datasets.MNIST("../data", train=True, download=True).targets.numpy()

X_test = datasets.MNIST("../data", train=False, download=True).data.numpy()
y_test = datasets.MNIST("../data", train=False, download=True).targets.numpy()

As a sanity check, we print out the size of the training and test data.

In [3]:
print('Training data shape: ', X_train.shape)
print('Training labels shape: ', y_train.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

Training data shape:  (60000, 28, 28)
Training labels shape:  (60000,)
Test data shape:  (10000, 28, 28)
Test labels shape:  (10000,)


We reshape the image data into rows. The datatype ```float``` allows you to subtract images (otherwise the datatype is ```uint8```)

In [4]:
X_train = np.reshape(X_train, (X_train.shape[0], -1)).astype('float')
X_test = np.reshape(X_test, (X_test.shape[0], -1)).astype('float')

print('Training data shape: ', X_train.shape)
print('Test data shape: ', X_test.shape)

Training data shape:  (60000, 784)
Test data shape:  (10000, 784)


We normalizing the input to make training easier (that's in general a good practice in Machine Learning)

In [5]:
X_mean = np.mean(X_train)
X_stddev = np.sqrt(np.var(X_train) + 1e-4)

X_train = (X_train - X_mean) / X_stddev
X_test = (X_test - X_mean) / X_stddev

Now we define the required Python functions we need later. Most of these functions you already saw in the assignment #4.

In [6]:
def batch(num):
    idx = np.random.randint(60000, size=num)
    return X_train[idx,:], y_train[idx]

def tanh(x):
    #TODO: Define the tanh activation function
    return np.tanh(x)

def softmax_crossentropy(a, y):
    an = (a.T - np.amax(a, axis=-1)).T
    return np.log(np.sum(np.exp(an), axis=-1)) - an[np.arange(a.shape[0]),y]

def softmax(a):
    an = (a.T - np.amax(a, axis=-1)).T
    expa = np.exp(an)
    return (expa.T/np.sum(expa, axis=-1)).T

def grad_softmax_crossentropy(a, y):
    s = softmax(a)
    s[np.arange(s.shape[0]),y] -= 1
    return s

Next we define the learning parameters.

In [7]:
training_steps = 10000
batch_size = 128
learning_rate = 0.1

Since we are using pure Python, we have to set up the network's weights and biases.

In [8]:
lim1 = np.sqrt(6 / (X_train.shape[-1] + 100))
W1 = np.random.uniform(low=-lim1, high=lim1, size=(X_train.shape[-1], 100))
b1 = np.zeros(100)

lim2 = np.sqrt(6 / (100 + 10))
W2 = np.random.uniform(low=-lim2, high=lim2, size=(100, 10))
b2 = np.zeros(10)

Now, we define the forward pass which calculates the predition and evaluates it. It finally returns the accuracy.

In [9]:
def evaluate():
    u1 = X_test @ W1 + b1
    z1 = tanh(u1)
    u2 = z1 @ W2 + b2
    pred = np.argmax(u2, axis=-1)
    acc = np.mean(pred == y_test)
    return acc

Finally the learning and your work comes into play. Fill out the **TODOs** in the code snippet below.

In [10]:
for step in range(training_steps):
    X_batch, y_batch = batch(batch_size)
    # TODO: Write down the dimension of X_batch and y_batch (...x...)
    # X_batch: 128x784 (batch size x nr. of pixels)
    # y_batch: 128x1 (batch size x label)

    # note: @ is matrix multiplication operation (np.matmul())
    u1 = X_batch @ W1 + b1
    # TODO: Write down the dimension of u1 (...x...)
    #u1: 128x100 (batch size x nr. of hidden units)

    z1 = tanh(u1)
    # TODO: Write down the dimension of z1 (...x...)
    #z1: 128x100 (batch size x nr. of hidden units)

    u2 = z1 @ W2 + b2
    # TODO: Write down the dimension of u2 (...x...)
    #u2: 128x10 (batch size x nr. of classes)

    loss = softmax_crossentropy(u2, y_batch) # value not actually needed, for debugging only
    # TODO: Write down the dimension of the loss (...x...)
    #loss: 128x1 (batch size x 1)

    dLdu2 = grad_softmax_crossentropy(u2, y_batch)
    # TODO: Write down the dimension of the dLdu2 (...x...)
    #dLdu2: 128x10 (batch size x nr. of classes)


    # TODO: implement the remaining backpropagation steps here
    delta2 = dLdu2
    dLdW2 = z1.T @ delta2
    dLdb2 = np.sum(delta2, axis=0)
    delta1 = dLdu2 @ W2.T * (1 - z1**2)
    dLdW1 = X_batch.T @ delta1
    dLdb1 = np.sum(delta1, axis=0)

    # update the weights
    assert(W1.shape == dLdW1.shape)
    assert(b1.shape == dLdb1.shape)
    assert(W2.shape == dLdW2.shape)
    assert(b2.shape == dLdb2.shape)
    W1 -= learning_rate * dLdW1
    b1 -= learning_rate * dLdb1
    W2 -= learning_rate * dLdW2
    b2 -= learning_rate * dLdb2

    if step % 500 == 0:
        print("step: {}, loss = {}, test acc = {}".format(step, np.mean(loss), evaluate()))
        # print(dLdb2.shape)

    # reduce learning rate at half and 3/4 of training steps
    if step == training_steps/2:
        learning_rate /= 10

step: 0, loss = 2.9377789022973047, test acc = 0.2164
step: 500, loss = 24.121173541492716, test acc = 0.613
step: 1000, loss = 37.66797805371061, test acc = 0.6656
step: 1500, loss = 28.31087136709026, test acc = 0.6581
step: 2000, loss = 19.715088012392314, test acc = 0.6533
step: 2500, loss = 12.737471066166432, test acc = 0.7808
step: 3000, loss = 38.598274150447, test acc = 0.7441
step: 3500, loss = 16.104118996584642, test acc = 0.693
step: 4000, loss = 10.74867545077144, test acc = 0.7512
step: 4500, loss = 16.95571749368045, test acc = 0.6359
step: 5000, loss = 7.6388967507057695, test acc = 0.793
step: 5500, loss = 5.889112897906552, test acc = 0.8413
step: 6000, loss = 2.620921838942344, test acc = 0.8426
step: 6500, loss = 3.555209675910491, test acc = 0.8409
step: 7000, loss = 3.188520304030413, test acc = 0.8363
step: 7500, loss = 1.689170149198394, test acc = 0.8355
step: 8000, loss = 2.80076284686439, test acc = 0.8304
step: 8500, loss = 2.2380288419999115, test acc = 0.

Let's evaluate the accuracy.

In [11]:
print("Test Accuracy: {}".format(evaluate()))

Test Accuracy: 0.8277
